# 06 · Figuras de la fase 1 para el póster

Este notebook **no estima nada**: lee la tabla de coeficientes que produjo `05_fase1` (`fase1_coeficientes_v2.csv`) y guarda en `resultados/figuras/` cuatro imágenes PNG (300 dpi):

1. `fase1_grafico_efectos.png`: efecto y su intervalo de 95 % por categoría. **A.** Efecto propio el día de la promoción. **B.** Otros SKUs de la misma clase (por +10 pp). **C.** Resto de la categoría (por +10 pp).
2. `fase1_tabla1_efecto_propio.png`, `fase1_tabla2_misma_clase.png`, `fase1_tabla3_otras_clases.png`: efecto del día y de los rezagos 1–3 y 4–6.

**Criterio de significancia.** En el efecto propio se usa el valor p; en los efectos de clase y categoría, el valor p ajustado por Benjamini‑Hochberg (`p_bh`, D‑15). Punto lleno o celda coloreada = significativo al 5 %.

In [ ]:
# Librerías
# ------------------------------------------------------------------------------------------------------------
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Parámetros
# ------------------------------------------------------------------------------------------------------------
categorias = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]
decimales = 1                                         # decimales de los porcentajes en las imágenes
color_neutro, color_negativo, color_positivo = "#1f5fa6", "#c0392b", "#27ae60"
color_significativo, color_cabecera = "#8fb5db", "#13213c"

ruta_tablas = os.path.join("resultados", "tablas")
ruta_figuras = os.path.join("resultados", "figuras")
os.makedirs(ruta_figuras, exist_ok=True)

# Leer los coeficientes y definir el valor p que decide la significancia de cada fila
coeficientes = pd.read_csv(os.path.join(ruta_tablas, "fase1_coeficientes_v2.csv"))
coeficientes["p_criterio"] = coeficientes["p_bh"].fillna(coeficientes["p_valor"])

## 1. Gráfico de efectos por categoría

In [ ]:
def datos_panel(variable: str) -> pd.DataFrame:
    """
    Esta función junta, para una variable, el efecto en % y su intervalo de 95 % por categoría. Las proporciones S
    se expresan por +10 puntos porcentuales.

    :param variable:    el nombre del coeficiente (p. ej., 'onpromotion').
    :return:            un DataFrame con una fila por categoría y las columnas efecto, inferior, superior y p.
    """

    d = coeficientes.loc[coeficientes["coeficiente"] == variable].set_index("categoria").loc[categorias]
    escala = 0.1 if variable.startswith("promo_") else 1.0
    return pd.DataFrame(
        {
            "efecto": 100 * np.expm1(escala * d["beta"]),
            "inferior": 100 * np.expm1(escala * d["ic_inf"]),
            "superior": 100 * np.expm1(escala * d["ic_sup"]),
            "p": d["p_criterio"],
        }
    )


paneles = [
    ("A. Efecto propio el día de la promoción", "onpromotion", "Cambio en la venta esperada (%)"),
    ("B. Otros SKUs de la misma clase", "promo_clase_otros", "Cambio por +10 pp de la clase\nen promoción (%)"),
    ("C. Resto de la categoría", "promo_cat_otras_clases", "Cambio por +10 pp de otras clases\nen promoción (%)"),
]

fig, ejes = plt.subplots(1, 3, figsize=(11.5, 4.5), sharey=True)
posiciones = np.arange(len(categorias))[::-1]   # la primera categoría arriba

for indice, (eje, (titulo, variable, etiqueta)) in enumerate(zip(ejes, paneles)):
    d = datos_panel(variable)
    for posicion, categoria in zip(posiciones, categorias):
        fila = d.loc[categoria]
        if indice == 0:
            color = color_neutro
        else:
            color = color_positivo if fila["efecto"] > 0 else color_negativo
        relleno = color if fila["p"] < 0.05 else "white"
        eje.hlines(posicion, fila["inferior"], fila["superior"], color=color, linewidth=2)
        eje.plot(fila["efecto"], posicion, "o", markersize=8, color=color, markerfacecolor=relleno, zorder=3)
        eje.annotate(f"{fila['efecto']:+.{decimales}f} %".replace(".", ","), (fila["efecto"], posicion), textcoords="offset points",
                     xytext=(0, -13), ha="center", fontsize=7.5, color="#444444")
    if indice > 0:
        eje.axvline(0, color="#555555", linewidth=1)
    eje.set_title(titulo, loc="left", fontsize=10.5, fontweight="bold")
    eje.set_xlabel(etiqueta, fontsize=8.5)
    eje.set_yticks(posiciones, categorias, fontsize=9)
    eje.set_ylim(-0.7, len(categorias) - 0.4)
    eje.grid(axis="x", color="#e5e5e5")
    eje.set_axisbelow(True)
    eje.spines[["top", "right", "left"]].set_visible(False)
    eje.tick_params(axis="y", length=0)

fig.text(0.01, 0.01, "Punto lleno: significativo al 5 % (B y C con valor p ajustado por Benjamini-Hochberg). "
         "IC 95 % con EE agrupados por SKU y semana. Tienda 44, 2016-2017.", fontsize=7.5, color="#777777")
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(os.path.join(ruta_figuras, "fase1_grafico_efectos.png"), dpi=300)
plt.show()

## 2. Tablas por tipo de promoción

Cada tabla tiene, por categoría, el efecto del día de la promoción y de los rezagos 1–3 y 4–6, con su valor p. Las celdas significativas al 5 % (con el criterio de arriba) van en azul.

In [ ]:
def guardar_tabla(variable: str, titulo: str, nombre_archivo: str) -> None:
    """
    Esta función dibuja la tabla de un tipo de promoción (día, rezago 1-3 y rezago 4-6) y la guarda como PNG.

    :param variable:        el nombre base del coeficiente (p. ej., 'onpromotion').
    :param titulo:          el título de la tabla.
    :param nombre_archivo:  el nombre del PNG, sin extensión.
    """

    c = coeficientes.set_index(["categoria", "coeficiente"])
    columna_efecto = "efecto_10pp_pct" if variable.startswith("promo_") else "efecto_pct"
    nombre_p = "p (BH)" if variable.startswith("promo_") else "p"

    # Armar los textos y los colores de las celdas
    textos, colores = [], []
    for categoria in categorias:
        fila_texto, fila_color = [categoria], ["#eef3f9"]
        for sufijo in ["", "_l1_3", "_l4_6"]:
            r = c.loc[(categoria, variable + sufijo)]
            p = r["p_criterio"]
            fila_texto += [f"{r[columna_efecto]:+.{decimales}f} %".replace(".", ","), "<0,001" if p < 0.001 else f"{p:.3f}".replace(".", ",")]
            fila_color += [color_significativo if p < 0.05 else "white"] * 2
        textos.append(fila_texto)
        colores.append(fila_color)

    cabeceras = ["Categoría", "Día de la promoción", nombre_p, "Rezago 1-3", nombre_p, "Rezago 4-6", nombre_p]
    fig, eje = plt.subplots(figsize=(11.5, 0.4 * (len(categorias) + 1)))
    eje.axis("off")
    anchos = [0.19] + [0.15, 0.12] * 3
    tabla = eje.table(cellText=textos, cellColours=colores, colLabels=cabeceras, colWidths=anchos, loc="center",
                      cellLoc="center")
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9.5)
    tabla.scale(1, 1.6)
    for j in range(len(cabeceras)):
        tabla[0, j].set_facecolor(color_cabecera)
        tabla[0, j].set_text_props(color="white", fontweight="bold")
    eje.set_title(titulo, loc="left", fontsize=11.5, fontweight="bold")
    fig.savefig(os.path.join(ruta_figuras, nombre_archivo + ".png"), dpi=300, bbox_inches="tight")
    plt.show()


guardar_tabla("onpromotion", "Tabla 1. Efecto propio de la promoción (variación % de la venta esperada del SKU)",
              "fase1_tabla1_efecto_propio")
guardar_tabla("promo_clase_otros", "Tabla 2. Otros SKUs de la misma clase (variación % por +10 pp en promoción)",
              "fase1_tabla2_misma_clase")
guardar_tabla("promo_cat_otras_clases", "Tabla 3. Otras clases de la categoría (variación % por +10 pp en promoción)",
              "fase1_tabla3_otras_clases")